## Example 1 — the installed version, and a function compiled to machine code gives Python's answer

In [ ]:
import importlib.metadata
import numba
import numpy as np

assert numba.__version__ == importlib.metadata.version("numba"), numba.__version__
@numba.njit
def collatz_steps(n):
    steps = 0
    while n != 1:
        n = n // 2 if n % 2 == 0 else 3 * n + 1
        steps += 1
    return steps
assert collatz_steps(27) == collatz_steps.py_func(27) == 111
assert len(collatz_steps.signatures) == 1 and collatz_steps.nopython_signatures, collatz_steps.signatures
values = np.arange(1, 2000)
assert np.array_equal(np.array([collatz_steps(v) for v in values[:50]]), np.array([collatz_steps.py_func(v) for v in values[:50]]))
print("numba", numba.__version__, "compiled", collatz_steps.signatures, "-> collatz_steps(27) =", collatz_steps(27))

## Example 2 — a parallel loop runs on numba's threading layer and matches NumPy

In [ ]:
import numba
import numpy as np

@numba.njit(parallel=True)
def row_norms(matrix):
    out = np.empty(matrix.shape[0])
    for i in numba.prange(matrix.shape[0]):
        total = 0.0
        for j in range(matrix.shape[1]):
            total += matrix[i, j] * matrix[i, j]
        out[i] = np.sqrt(total)
    return out
matrix = np.random.default_rng(0).standard_normal((2000, 64))
assert np.allclose(row_norms(matrix), np.linalg.norm(matrix, axis=1))
layer = numba.threading_layer()
assert layer in ("tbb", "omp", "workqueue"), layer
print("2000 row norms in parallel on the", layer, "layer with", numba.get_num_threads(), "threads")

## Example 3 — compiled universal functions: `vectorize` and `guvectorize`

In [ ]:
import numba
import numpy as np

@numba.vectorize(["float32(float32, float32)", "float64(float64, float64)"])   # narrowest type first
def hypot(x, y):
    return (x * x + y * y) ** 0.5
@numba.guvectorize(["void(float64[:], float64[:])"], "(n)->()")
def spread(values, out):
    out[0] = values.max() - values.min()
x = np.linspace(0.0, 3.0, 7)
assert np.allclose(hypot(x, 4.0), np.hypot(x, 4.0))
single = hypot(np.full(3, 3, dtype=np.float32), np.full(3, 4, dtype=np.float32))
assert single.dtype == np.float32 and np.array_equal(single, np.full(3, 5, dtype=np.float32)), (single.dtype, single)
grid = np.arange(12, dtype=np.float64).reshape(3, 4)
assert np.array_equal(spread(grid), np.array([3.0, 3.0, 3.0]))
assert np.allclose(hypot.reduce(np.array([3.0, 4.0, 12.0])), 13.0)       # a real ufunc: it has reduce()
print("vectorize broadcasts and reduces; guvectorize maps", grid.shape, "->", spread(grid).shape)

## Example 4 — a jitted matrix product, a compiled class and a typed dictionary

In [ ]:
import numba
import numpy as np
from numba.experimental import jitclass
from numba.typed import Dict, List

@numba.njit
def gram(a):
    return a @ a.T                      # compiled BLAS call through SciPy
a = np.random.default_rng(1).standard_normal((40, 30))
assert np.allclose(gram(a), a @ a.T)
@jitclass([("total", numba.float64), ("count", numba.int64)])
class Mean:
    def __init__(self):
        self.total, self.count = 0.0, 0
    def add(self, value):
        self.total += value
        self.count += 1
        return self.total / self.count
@numba.njit
def fill(mean, values, seen):
    for value in values:
        seen[value] = mean.add(value)
seen = Dict.empty(numba.float64, numba.float64)
fill(Mean(), List([2.0, 4.0, 9.0]), seen)
assert dict(seen) == {2.0: 2.0, 4.0: 3.0, 9.0: 5.0}, dict(seen)
print("jitted a @ a.T matches NumPy; a compiled class filled a typed dictionary:", dict(seen))

## Example 5 — code numba cannot compile is refused with a typing error, and a compiled function can be cached on disk

In [ ]:
import os
import subprocess
import sys
import tempfile
import numba

@numba.njit
def mixed(flag):
    return 1 if flag else "one"
try:
    mixed(True)
    raise AssertionError("a function returning two unrelated types was compiled")
except numba.core.errors.TypingError as error:
    print("refused:", str(error).strip().splitlines()[0][:110])
with tempfile.TemporaryDirectory() as folder:
    with open(os.path.join(folder, "cached.py"), "w") as handle:
        handle.write("import numba\n@numba.njit(cache=True)\ndef triple(x):\n    return 3 * x\nprint(triple(14), triple.stats.cache_hits)\n")
    environment = {**os.environ, "NUMBA_CACHE_DIR": os.path.join(folder, "cache")}
    runs = [subprocess.run([sys.executable, "cached.py"], cwd=folder, env=environment, capture_output=True, text=True, timeout=600)
            for _ in range(2)]
    assert [run.returncode for run in runs] == [0, 0], runs[-1].stderr[-1500:]
    first, second = (run.stdout.split(maxsplit=1) for run in runs)
assert first[0] == second[0] == "42" and "}" in second[1] and second[1].strip() != first[1].strip(), (first, second)
print("first run compiled (cache hits", first[1].strip() + "), second run loaded from disk (cache hits", second[1].strip() + ")")